# Final Decomposed PICO Extraction with Bio_ClinicalBERT

This notebook presents a **complete decomposed method** for extracting a PICO table from **EBM-NLP 2.00**.

The method is decomposed into two stages:

1. **Sentence-level filtering**: decide whether a sentence is relevant for Participants, Interventions, or Outcomes
2. **Token-level fine-grained extraction**: extract subtype-specific spans only from the relevant sentences

We evaluate both stages with:

- accuracy
- precision
- recall
- F1

We run both:

- **zero-shot**
- **few-shot**

using:

- `emilyalsentzer/Bio_ClinicalBERT`

The goal of this notebook is not only to produce results, but also to make every step understandable to a reader such as the professor.


## Study Goal

The earlier version of the decomposed pipeline had a strong sentence stage but a weak token stage.

The main hypothesis in this notebook is:

1. sentence filtering can work reasonably well
2. token extraction suffers if we use only coarse BIO labels
3. token extraction should improve when we use the **fine-grained hierarchical labels** already available in EBM-NLP:
   - Participants: Age, Sex, Sample size, Condition
   - Interventions: Surgical, Physical, Drug, Educational, Psychological, Other, Control
   - Outcomes: Physical, Pain, Mortality, Adverse effects, Mental, Other

So this notebook explicitly uses those subtype labels and tracks whether the bottleneck is:

- the sentence stage
- the token stage
- or the interaction between both stages


## 1. Setup

This notebook is **self-contained**.

It does not need to import the project `.py` helper files during notebook execution.

Instead, the full working source for the token-level and decomposed pipelines is embedded directly inside notebook code cells and loaded into in-memory modules.

If packages are missing, install them first:

```python
!pip install transformers datasets seqeval scikit-learn pandas seaborn matplotlib torch
```


In [ ]:
from __future__ import annotations

import json
import sys
import types
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

PROJECT_DIR = Path("/Users/priyanshugurjar/Downloads/AI AND TEXT/ebm_nlp_2_00")


def load_inline_module(module_name: str, source: str, file_path: Path, extra_globals=None):
    module = types.ModuleType(module_name)
    module.__file__ = str(file_path)
    if extra_globals:
        module.__dict__.update(extra_globals)
    sys.modules[module_name] = module
    exec(compile(source, str(file_path), "exec"), module.__dict__)
    return module


CTE_SOURCE = '#!/usr/bin/env python3\n"""\nFine-grained token-level entity extraction on EBM-NLP with Bio_ClinicalBERT.\n\nThis script supports two regimes with the same encoder:\n\n1. zero-shot\n   A label-prototype baseline that uses Bio_ClinicalBERT contextual embeddings\n   without supervised fine-tuning.\n\n2. few-shot\n   Supervised token classification fine-tuning on a small subset of training\n   documents.\n\nThe EBM-NLP copy in this workspace stores fine-grained token labels as integer\nIDs in:\n  annotations/aggregated/hierarchical_labels/<entity_type>/<split>/\n\nThose IDs are converted into BIO-with-type tags such as:\n  O, B-AGE, I-AGE, B-DRUG, I-DRUG, ...\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport random\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Mapping, MutableMapping, Sequence, Tuple\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom datasets import Dataset\nfrom seqeval.metrics import (\n    accuracy_score as seqeval_accuracy_score,\n    classification_report as seqeval_classification_report,\n    f1_score as seqeval_f1_score,\n    precision_score as seqeval_precision_score,\n    recall_score as seqeval_recall_score,\n)\nfrom transformers import (\n    AutoModel,\n    AutoModelForTokenClassification,\n    AutoTokenizer,\n    DataCollatorForTokenClassification,\n    Trainer,\n    TrainingArguments,\n    modeling_utils as transformers_modeling_utils,\n)\n\n\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\n\ntry:\n    import transformers.safetensors_conversion as safetensors_conversion\n\n    def _disable_auto_conversion(*args, **kwargs):\n        return None, None, None\n\n    safetensors_conversion.auto_conversion = _disable_auto_conversion\n    transformers_modeling_utils.auto_conversion = _disable_auto_conversion\nexcept Exception:\n    pass\n\nMODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"\nENTITY_TYPES = ("participants", "interventions", "outcomes")\nSENT_END = {".", "!", "?"}\nFUNCTION_WORDS = {\n    "a",\n    "an",\n    "and",\n    "are",\n    "as",\n    "at",\n    "be",\n    "by",\n    "for",\n    "from",\n    "in",\n    "into",\n    "is",\n    "of",\n    "on",\n    "or",\n    "that",\n    "the",\n    "their",\n    "to",\n    "was",\n    "were",\n    "with",\n}\n\n\nENTITY_LABELS: Dict[str, Dict[int, str]] = {\n    "participants": {\n        0: "No label",\n        1: "Age",\n        2: "Sex",\n        3: "Sample size",\n        4: "Condition",\n    },\n    "interventions": {\n        0: "No label",\n        1: "Surgical",\n        2: "Physical",\n        3: "Drug",\n        4: "Educational",\n        5: "Psychological",\n        6: "Other",\n        7: "Control",\n    },\n    "outcomes": {\n        0: "No label",\n        1: "Physical",\n        2: "Pain",\n        3: "Mortality",\n        4: "Adverse effects",\n        5: "Mental",\n        6: "Other",\n    },\n}\n\n\nZERO_SHOT_PROMPTS: Dict[str, Dict[int, List[str]]] = {\n    "participants": {\n        0: [\n            "background token outside any participant detail",\n            "word not describing participant characteristics",\n        ],\n        1: [\n            "participant age",\n            "age of the enrolled patients",\n            "mean age or age range of participants",\n        ],\n        2: [\n            "participant sex",\n            "sex or gender of participants",\n            "male or female participants",\n        ],\n        3: [\n            "sample size",\n            "number of patients in the study",\n            "count of enrolled participants",\n        ],\n        4: [\n            "participant medical condition",\n            "disease or diagnosis of the patients",\n            "health condition of participants",\n        ],\n    },\n    "interventions": {\n        0: [\n            "background token outside any intervention detail",\n            "word not describing a treatment or control arm",\n        ],\n        1: [\n            "surgical intervention",\n            "surgical procedure or operation",\n            "operative treatment",\n        ],\n        2: [\n            "physical intervention",\n            "exercise therapy or physical treatment",\n            "rehabilitation or device based intervention",\n        ],\n        3: [\n            "drug intervention",\n            "medication or pharmacological treatment",\n            "dose of a medicine or drug therapy",\n        ],\n        4: [\n            "educational intervention",\n            "patient education or counselling program",\n            "training or instructional intervention",\n        ],\n        5: [\n            "psychological intervention",\n            "behavioral therapy or psychotherapy",\n            "mental health treatment",\n        ],\n        6: [\n            "other intervention",\n            "miscellaneous treatment not covered by the other groups",\n        ],\n        7: [\n            "control group",\n            "placebo arm or usual care comparator",\n            "standard care control treatment",\n        ],\n    },\n    "outcomes": {\n        0: [\n            "background token outside any outcome detail",\n            "word not describing a measured study outcome",\n        ],\n        1: [\n            "physical outcome",\n            "physiological measurement or physical function outcome",\n            "clinical physical endpoint",\n        ],\n        2: [\n            "pain outcome",\n            "pain score or pain measurement",\n            "pain intensity outcome",\n        ],\n        3: [\n            "mortality outcome",\n            "death or survival outcome",\n            "mortality endpoint",\n        ],\n        4: [\n            "adverse effects",\n            "side effect or treatment harm",\n            "complication or adverse event outcome",\n        ],\n        5: [\n            "mental outcome",\n            "psychological or psychiatric outcome",\n            "depression anxiety or mental health measure",\n        ],\n        6: [\n            "other outcome",\n            "clinical outcome not covered by the other groups",\n        ],\n    },\n}\n\n\n@dataclass\nclass Document:\n    doc_id: str\n    tokens: List[str]\n    label_ids: List[int]\n    tags: List[str]\n\n\n@dataclass\nclass SentenceExample:\n    doc_id: str\n    sent_idx: int\n    tokens: List[str]\n    tags: List[str]\n    orig_indices: List[int]\n\n\ndef set_seed(seed: int) -> None:\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef to_jsonable(value):\n    if isinstance(value, dict):\n        return {str(key): to_jsonable(val) for key, val in value.items()}\n    if isinstance(value, list):\n        return [to_jsonable(item) for item in value]\n    if isinstance(value, tuple):\n        return [to_jsonable(item) for item in value]\n    if isinstance(value, np.generic):\n        return value.item()\n    if isinstance(value, torch.Tensor):\n        return value.detach().cpu().tolist()\n    if isinstance(value, Path):\n        return str(value)\n    return value\n\n\ndef get_device(explicit_device: str | None = None) -> torch.device:\n    if explicit_device:\n        return torch.device(explicit_device)\n    if torch.cuda.is_available():\n        return torch.device("cuda")\n    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():\n        return torch.device("mps")\n    return torch.device("cpu")\n\n\ndef sanitize_label_name(name: str) -> str:\n    return name.upper().replace(" ", "_")\n\n\ndef entity_tag_name_map(entity_type: str) -> Dict[int, str]:\n    return {\n        label_id: sanitize_label_name(label_name)\n        for label_id, label_name in ENTITY_LABELS[entity_type].items()\n        if label_id != 0\n    }\n\n\ndef build_label_list(entity_type: str) -> List[str]:\n    tags = ["O"]\n    for label_id in sorted(entity_tag_name_map(entity_type)):\n        tag_name = entity_tag_name_map(entity_type)[label_id]\n        tags.append(f"B-{tag_name}")\n        tags.append(f"I-{tag_name}")\n    return tags\n\n\ndef get_data_dir(cli_value: str | None) -> Path:\n    if cli_value:\n        return Path(cli_value).expanduser().resolve()\n\n    script_dir = Path(__file__).resolve().parent\n    if (script_dir / "documents").exists() and (script_dir / "annotations").exists():\n        return script_dir\n\n    cwd = Path.cwd().resolve()\n    if (cwd / "documents").exists() and (cwd / "annotations").exists():\n        return cwd\n    raise FileNotFoundError(\n        "Could not locate the dataset. Pass --data-dir /path/to/ebm_nlp_2_00."\n    )\n\n\ndef get_doc_ids(data_dir: Path, split: str, entity_type: str) -> List[str]:\n    split_dir = "test/gold" if split == "test" else "train"\n    ann_dir = (\n        data_dir\n        / "annotations"\n        / "aggregated"\n        / "hierarchical_labels"\n        / entity_type\n        / split_dir\n    )\n    return sorted(path.stem.split(".")[0] for path in ann_dir.glob("*.AGGREGATED.ann"))\n\n\ndef load_tokens(data_dir: Path, doc_id: str) -> List[str]:\n    path = data_dir / "documents" / f"{doc_id}.tokens"\n    return path.read_text(encoding="utf-8").splitlines()\n\n\ndef load_label_ids(data_dir: Path, doc_id: str, entity_type: str, split: str) -> List[int]:\n    split_dir = "test/gold" if split == "test" else "train"\n    path = (\n        data_dir\n        / "annotations"\n        / "aggregated"\n        / "hierarchical_labels"\n        / entity_type\n        / split_dir\n        / f"{doc_id}.AGGREGATED.ann"\n    )\n    return [int(line) for line in path.read_text(encoding="utf-8").splitlines()]\n\n\ndef hierarchical_to_fine_bio(label_ids: Sequence[int], entity_type: str) -> List[str]:\n    tag_name_map = entity_tag_name_map(entity_type)\n    tags: List[str] = []\n    prev_label = 0\n    for label_id in label_ids:\n        if label_id == 0:\n            tags.append("O")\n            prev_label = 0\n            continue\n        prefix = "I" if prev_label == label_id else "B"\n        tags.append(f"{prefix}-{tag_name_map[label_id]}")\n        prev_label = label_id\n    return tags\n\n\ndef load_dataset(\n    data_dir: Path,\n    entity_type: str,\n    split: str,\n    limit: int | None = None,\n) -> List[Document]:\n    documents: List[Document] = []\n    for doc_id in get_doc_ids(data_dir, split, entity_type):\n        tokens = load_tokens(data_dir, doc_id)\n        label_ids = load_label_ids(data_dir, doc_id, entity_type, split)\n        if len(tokens) != len(label_ids):\n            raise ValueError(\n                f"Length mismatch for {entity_type}/{split}/{doc_id}: "\n                f"{len(tokens)} tokens vs {len(label_ids)} labels"\n            )\n        documents.append(\n            Document(\n                doc_id=doc_id,\n                tokens=tokens,\n                label_ids=label_ids,\n                tags=hierarchical_to_fine_bio(label_ids, entity_type),\n            )\n        )\n        if limit is not None and len(documents) >= limit:\n            break\n    return documents\n\n\ndef split_into_sentences(\n    tokens: Sequence[str],\n    tags: Sequence[str],\n    max_tokens: int = 200,\n) -> List[Tuple[List[str], List[str], List[int]]]:\n    chunks: List[Tuple[List[str], List[str], List[int]]] = []\n    cur_tokens: List[str] = []\n    cur_tags: List[str] = []\n    cur_indices: List[int] = []\n\n    for idx, (token, tag) in enumerate(zip(tokens, tags)):\n        cur_tokens.append(token)\n        cur_tags.append(tag)\n        cur_indices.append(idx)\n\n        should_flush = token in SENT_END or len(cur_tokens) >= max_tokens\n        if should_flush:\n            chunks.append((cur_tokens[:], cur_tags[:], cur_indices[:]))\n            cur_tokens.clear()\n            cur_tags.clear()\n            cur_indices.clear()\n\n    if cur_tokens:\n        chunks.append((cur_tokens[:], cur_tags[:], cur_indices[:]))\n\n    return chunks\n\n\ndef docs_to_sentences(documents: Sequence[Document], max_tokens: int = 200) -> List[SentenceExample]:\n    sentence_examples: List[SentenceExample] = []\n    for doc in documents:\n        chunks = split_into_sentences(doc.tokens, doc.tags, max_tokens=max_tokens)\n        for sent_idx, (sent_tokens, sent_tags, indices) in enumerate(chunks):\n            if sent_tokens:\n                sentence_examples.append(\n                    SentenceExample(\n                        doc_id=doc.doc_id,\n                        sent_idx=sent_idx,\n                        tokens=sent_tokens,\n                        tags=sent_tags,\n                        orig_indices=indices,\n                    )\n                )\n    return sentence_examples\n\n\ndef split_train_dev(\n    examples: Sequence[SentenceExample],\n    dev_ratio: float,\n    seed: int,\n) -> Tuple[List[SentenceExample], List[SentenceExample]]:\n    items = list(examples)\n    if len(items) <= 1:\n        return items, []\n    rng = random.Random(seed)\n    rng.shuffle(items)\n    dev_size = max(1, int(round(len(items) * dev_ratio)))\n    dev_size = min(dev_size, len(items) - 1)\n    dev_items = items[:dev_size]\n    train_items = items[dev_size:]\n    return train_items, dev_items\n\n\ndef collect_doc_label_types(doc: Document) -> set[str]:\n    return {tag[2:] for tag in doc.tags if tag != "O"}\n\n\ndef sample_few_shot_docs(\n    train_docs: Sequence[Document],\n    shots: int,\n    seed: int,\n) -> List[Document]:\n    docs = list(train_docs)\n    if shots >= len(docs):\n        return docs\n\n    rng = random.Random(seed)\n    label_frequency = Counter()\n    doc_label_sets = [collect_doc_label_types(doc) for doc in docs]\n    for label_set in doc_label_sets:\n        label_frequency.update(label_set)\n\n    selected_indices: List[int] = []\n    available_indices = set(range(len(docs)))\n    covered_labels: set[str] = set()\n\n    for label_name, _ in sorted(label_frequency.items(), key=lambda item: (item[1], item[0])):\n        candidates = [\n            idx for idx in available_indices if label_name in doc_label_sets[idx]\n        ]\n        if not candidates:\n            continue\n        candidates.sort(\n            key=lambda idx: (\n                -(len(doc_label_sets[idx] - covered_labels)),\n                docs[idx].doc_id,\n            )\n        )\n        chosen = candidates[0]\n        selected_indices.append(chosen)\n        available_indices.remove(chosen)\n        covered_labels.update(doc_label_sets[chosen])\n        if len(selected_indices) >= shots:\n            break\n\n    remaining = list(available_indices)\n    rng.shuffle(remaining)\n    selected_indices.extend(remaining[: max(0, shots - len(selected_indices))])\n    selected_indices = sorted(selected_indices[:shots])\n    return [docs[idx] for idx in selected_indices]\n\n\ndef sentence_examples_to_dataset(\n    sentence_examples: Sequence[SentenceExample],\n    label_to_id: Mapping[str, int],\n) -> Dataset:\n    return Dataset.from_dict(\n        {\n            "doc_id": [ex.doc_id for ex in sentence_examples],\n            "sent_idx": [ex.sent_idx for ex in sentence_examples],\n            "tokens": [ex.tokens for ex in sentence_examples],\n            "ner_tags": [[label_to_id[tag] for tag in ex.tags] for ex in sentence_examples],\n        }\n    )\n\n\ndef build_tokenize_and_align_labels(tokenizer, label_to_id: Mapping[str, int], id_to_label: Mapping[int, str]):\n    def tokenize_and_align_labels(batch: MutableMapping[str, List[List[str]]]) -> MutableMapping[str, List[List[int]]]:\n        tokenized = tokenizer(\n            batch["tokens"],\n            truncation=True,\n            is_split_into_words=True,\n            max_length=256,\n        )\n        aligned_labels: List[List[int]] = []\n\n        for i, labels in enumerate(batch["ner_tags"]):\n            word_ids = tokenized.word_ids(batch_index=i)\n            previous_word_idx = None\n            label_ids: List[int] = []\n            for word_idx in word_ids:\n                if word_idx is None:\n                    label_ids.append(-100)\n                elif word_idx != previous_word_idx:\n                    label_ids.append(labels[word_idx])\n                else:\n                    raw_label = id_to_label[labels[word_idx]]\n                    continued_label = (\n                        f"I-{raw_label[2:]}" if raw_label.startswith("B-") else raw_label\n                    )\n                    label_ids.append(label_to_id[continued_label])\n                previous_word_idx = word_idx\n\n            aligned_labels.append(label_ids)\n\n        tokenized["labels"] = aligned_labels\n        return tokenized\n\n    return tokenize_and_align_labels\n\n\ndef train_few_shot_model(\n    model_name: str,\n    train_sentences: Sequence[SentenceExample],\n    dev_sentences: Sequence[SentenceExample],\n    label_list: Sequence[str],\n    output_dir: Path,\n    epochs: int,\n    batch_size: int,\n    learning_rate: float,\n    seed: int,\n    local_files_only: bool,\n):\n    label_to_id = {label: idx for idx, label in enumerate(label_list)}\n    id_to_label = {idx: label for label, idx in label_to_id.items()}\n\n    tokenizer = AutoTokenizer.from_pretrained(\n        model_name,\n        local_files_only=local_files_only,\n    )\n    train_ds = sentence_examples_to_dataset(train_sentences, label_to_id)\n    dev_ds = sentence_examples_to_dataset(dev_sentences, label_to_id) if dev_sentences else None\n    mapper = build_tokenize_and_align_labels(tokenizer, label_to_id, id_to_label)\n\n    train_tok = train_ds.map(\n        mapper,\n        batched=True,\n        remove_columns=train_ds.column_names,\n    )\n    dev_tok = (\n        dev_ds.map(\n            mapper,\n            batched=True,\n            remove_columns=dev_ds.column_names,\n        )\n        if dev_ds is not None\n        else None\n    )\n\n    model = AutoModelForTokenClassification.from_pretrained(\n        model_name,\n        num_labels=len(label_list),\n        id2label=id_to_label,\n        label2id=label_to_id,\n        local_files_only=local_files_only,\n        use_safetensors=False,\n    )\n\n    args = TrainingArguments(\n        output_dir=str(output_dir),\n        learning_rate=learning_rate,\n        per_device_train_batch_size=batch_size,\n        per_device_eval_batch_size=batch_size,\n        num_train_epochs=epochs,\n        weight_decay=0.01,\n        optim="adamw_torch",\n        eval_strategy="epoch" if dev_tok is not None else "no",\n        save_strategy="no",\n        logging_strategy="epoch",\n        report_to="none",\n        remove_unused_columns=False,\n        seed=seed,\n        data_seed=seed,\n        fp16=torch.cuda.is_available(),\n    )\n\n    trainer = Trainer(\n        model=model,\n        args=args,\n        train_dataset=train_tok,\n        eval_dataset=dev_tok,\n        processing_class=tokenizer,\n        data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),\n    )\n    trainer.train()\n    return tokenizer, model, label_to_id, id_to_label\n\n\n@torch.no_grad()\ndef predict_few_shot_tags(\n    tokenizer,\n    model,\n    sentence_examples: Sequence[SentenceExample],\n    id_to_label: Mapping[int, str],\n    device: torch.device,\n) -> Dict[Tuple[str, int], List[str]]:\n    model.to(device)\n    model.eval()\n    predictions: Dict[Tuple[str, int], List[str]] = {}\n\n    for example in sentence_examples:\n        encoding = tokenizer(\n            example.tokens,\n            is_split_into_words=True,\n            return_tensors="pt",\n            truncation=True,\n            max_length=256,\n        )\n        inputs = {key: value.to(device) for key, value in encoding.items()}\n        logits = model(**inputs).logits[0].detach().cpu()\n        pred_ids = logits.argmax(dim=-1).tolist()\n        word_ids = encoding.word_ids(batch_index=0)\n\n        sentence_preds: List[str] = []\n        previous_word_idx = None\n        for pred_id, word_idx in zip(pred_ids, word_ids):\n            if word_idx is None or word_idx == previous_word_idx:\n                continue\n            sentence_preds.append(id_to_label[pred_id])\n            previous_word_idx = word_idx\n\n        if len(sentence_preds) < len(example.tokens):\n            sentence_preds.extend(["O"] * (len(example.tokens) - len(sentence_preds)))\n        predictions[(example.doc_id, example.sent_idx)] = sentence_preds[: len(example.tokens)]\n\n    return predictions\n\n\nclass ZeroShotPrototypeTagger:\n    def __init__(\n        self,\n        model_name: str,\n        device: torch.device,\n        zero_shot_margin: float,\n        zero_shot_threshold: float,\n        local_files_only: bool,\n    ) -> None:\n        self.model_name = model_name\n        self.device = device\n        self.zero_shot_margin = zero_shot_margin\n        self.zero_shot_threshold = zero_shot_threshold\n        self.local_files_only = local_files_only\n        self.tokenizer = AutoTokenizer.from_pretrained(\n            model_name,\n            local_files_only=local_files_only,\n        )\n        self.model = AutoModel.from_pretrained(\n            model_name,\n            local_files_only=local_files_only,\n            use_safetensors=False,\n        ).to(device)\n        self.model.eval()\n        self.prototype_cache: Dict[str, Tuple[List[int], torch.Tensor]] = {}\n\n    @torch.no_grad()\n    def _mean_last_four_layers(self, model_outputs) -> torch.Tensor:\n        hidden_stack = torch.stack(model_outputs.hidden_states[-4:], dim=0)\n        return hidden_stack.mean(dim=0)\n\n    @torch.no_grad()\n    def _encode_prototype_text(self, text: str) -> torch.Tensor:\n        encoding = self.tokenizer(\n            text,\n            return_tensors="pt",\n            truncation=True,\n            max_length=32,\n            return_special_tokens_mask=True,\n        )\n        inputs = {\n            key: value.to(self.device)\n            for key, value in encoding.items()\n            if key != "special_tokens_mask"\n        }\n        outputs = self.model(**inputs, output_hidden_states=True)\n        hidden = self._mean_last_four_layers(outputs)[0]\n        special_mask = encoding["special_tokens_mask"][0].bool().to(self.device)\n        usable = hidden[~special_mask]\n        if usable.numel() == 0:\n            usable = hidden\n        return F.normalize(usable.mean(dim=0), dim=0)\n\n    @torch.no_grad()\n    def _build_prototypes(self, entity_type: str) -> Tuple[List[int], torch.Tensor]:\n        if entity_type in self.prototype_cache:\n            return self.prototype_cache[entity_type]\n\n        label_ids = sorted(ZERO_SHOT_PROMPTS[entity_type])\n        prototype_vectors: List[torch.Tensor] = []\n        for label_id in label_ids:\n            prompt_vectors = [\n                self._encode_prototype_text(prompt)\n                for prompt in ZERO_SHOT_PROMPTS[entity_type][label_id]\n            ]\n            prototype = F.normalize(torch.stack(prompt_vectors, dim=0).mean(dim=0), dim=0)\n            prototype_vectors.append(prototype)\n\n        prototypes = torch.stack(prototype_vectors, dim=0)\n        self.prototype_cache[entity_type] = (label_ids, prototypes)\n        return label_ids, prototypes\n\n    @torch.no_grad()\n    def predict_sentence(self, entity_type: str, tokens: Sequence[str]) -> List[str]:\n        label_ids, prototypes = self._build_prototypes(entity_type)\n        o_index = label_ids.index(0)\n        tag_name_map = entity_tag_name_map(entity_type)\n\n        encoding = self.tokenizer(\n            list(tokens),\n            is_split_into_words=True,\n            return_tensors="pt",\n            truncation=True,\n            max_length=256,\n        )\n        inputs = {key: value.to(self.device) for key, value in encoding.items()}\n        outputs = self.model(**inputs, output_hidden_states=True)\n        hidden = self._mean_last_four_layers(outputs)[0]\n        word_ids = encoding.word_ids(batch_index=0)\n\n        word_vectors: List[torch.Tensor] = []\n        current_word_idx = None\n        bucket: List[torch.Tensor] = []\n        for token_vec, word_idx in zip(hidden, word_ids):\n            if word_idx is None:\n                continue\n            if current_word_idx is None:\n                current_word_idx = word_idx\n            if word_idx != current_word_idx:\n                word_vectors.append(torch.stack(bucket, dim=0).mean(dim=0))\n                bucket = [token_vec]\n                current_word_idx = word_idx\n            else:\n                bucket.append(token_vec)\n        if bucket:\n            word_vectors.append(torch.stack(bucket, dim=0).mean(dim=0))\n\n        if not word_vectors:\n            return ["O"] * len(tokens)\n\n        token_matrix = F.normalize(torch.stack(word_vectors, dim=0), dim=1)\n        score_matrix = token_matrix @ prototypes.T\n        raw_best_indices = score_matrix.argmax(dim=1).tolist()\n        predicted_label_ids: List[int] = []\n\n        for token, best_idx, scores in zip(tokens, raw_best_indices, score_matrix):\n            best_label_id = label_ids[best_idx]\n            best_score = float(scores[best_idx])\n            o_score = float(scores[o_index])\n            margin = best_score - o_score\n\n            if not any(ch.isalnum() for ch in token):\n                predicted_label_ids.append(0)\n                continue\n\n            if (\n                best_label_id != 0\n                and token.lower() in FUNCTION_WORDS\n                and margin < max(self.zero_shot_margin, 0.08)\n            ):\n                predicted_label_ids.append(0)\n                continue\n\n            if best_label_id != 0 and (\n                best_score < self.zero_shot_threshold or margin < self.zero_shot_margin\n            ):\n                predicted_label_ids.append(0)\n                continue\n\n            predicted_label_ids.append(best_label_id)\n\n        if len(predicted_label_ids) < len(tokens):\n            predicted_label_ids.extend([0] * (len(tokens) - len(predicted_label_ids)))\n        predicted_label_ids = predicted_label_ids[: len(tokens)]\n\n        tags: List[str] = []\n        prev_label_id = 0\n        for label_id in predicted_label_ids:\n            if label_id == 0:\n                tags.append("O")\n                prev_label_id = 0\n                continue\n            prefix = "I" if prev_label_id == label_id else "B"\n            tags.append(f"{prefix}-{tag_name_map[label_id]}")\n            prev_label_id = label_id\n        return tags\n\n\ndef run_zero_shot(\n    entity_type: str,\n    test_sentences: Sequence[SentenceExample],\n    tagger: ZeroShotPrototypeTagger,\n) -> Dict[Tuple[str, int], List[str]]:\n    predictions: Dict[Tuple[str, int], List[str]] = {}\n    for example in test_sentences:\n        predictions[(example.doc_id, example.sent_idx)] = tagger.predict_sentence(\n            entity_type,\n            example.tokens,\n        )\n    return predictions\n\n\ndef evaluate_predictions(\n    gold_sentences: Sequence[SentenceExample],\n    pred_map: Mapping[Tuple[str, int], Sequence[str]],\n) -> Dict[str, object]:\n    gold_sequences: List[List[str]] = []\n    pred_sequences: List[List[str]] = []\n    token_correct = 0\n    token_total = 0\n\n    for example in gold_sentences:\n        gold_tags = list(example.tags)\n        pred_tags = list(pred_map.get((example.doc_id, example.sent_idx), ["O"] * len(gold_tags)))\n        if len(pred_tags) < len(gold_tags):\n            pred_tags.extend(["O"] * (len(gold_tags) - len(pred_tags)))\n        pred_tags = pred_tags[: len(gold_tags)]\n\n        gold_sequences.append(gold_tags)\n        pred_sequences.append(pred_tags)\n        token_correct += sum(gold == pred for gold, pred in zip(gold_tags, pred_tags))\n        token_total += len(gold_tags)\n\n    report = seqeval_classification_report(\n        gold_sequences,\n        pred_sequences,\n        mode="strict",\n        scheme=None,\n        output_dict=True,\n        zero_division=0,\n    )\n    return {\n        "precision": float(\n            seqeval_precision_score(gold_sequences, pred_sequences, zero_division=0)\n        ),\n        "recall": float(\n            seqeval_recall_score(gold_sequences, pred_sequences, zero_division=0)\n        ),\n        "f1": float(seqeval_f1_score(gold_sequences, pred_sequences, zero_division=0)),\n        "entity_accuracy": float(seqeval_accuracy_score(gold_sequences, pred_sequences)),\n        "token_accuracy": (token_correct / token_total) if token_total else 0.0,\n        "support_sentences": len(gold_sequences),\n        "support_tokens": token_total,\n        "classification_report": report,\n    }\n\n\ndef save_predictions(\n    output_path: Path,\n    sentence_examples: Sequence[SentenceExample],\n    pred_map: Mapping[Tuple[str, int], Sequence[str]],\n) -> None:\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    with output_path.open("w", encoding="utf-8") as handle:\n        for example in sentence_examples:\n            pred_tags = list(pred_map.get((example.doc_id, example.sent_idx), ["O"] * len(example.tokens)))\n            if len(pred_tags) < len(example.tokens):\n                pred_tags.extend(["O"] * (len(example.tokens) - len(pred_tags)))\n            row = {\n                "doc_id": example.doc_id,\n                "sent_idx": example.sent_idx,\n                "tokens": example.tokens,\n                "gold_tags": example.tags,\n                "pred_tags": pred_tags[: len(example.tokens)],\n            }\n            handle.write(json.dumps(row) + "\\n")\n\n\ndef run_entity_type(\n    data_dir: Path,\n    entity_type: str,\n    strategy: str,\n    model_name: str,\n    output_dir: Path,\n    few_shot_docs: int,\n    epochs: int,\n    batch_size: int,\n    learning_rate: float,\n    seed: int,\n    device: torch.device,\n    zero_shot_margin: float,\n    zero_shot_threshold: float,\n    local_files_only: bool,\n    max_train_docs: int | None,\n    max_eval_docs: int | None,\n) -> Dict[str, object]:\n    train_docs = load_dataset(data_dir, entity_type, "train", limit=max_train_docs)\n    test_docs = load_dataset(data_dir, entity_type, "test", limit=max_eval_docs)\n    test_sentences = docs_to_sentences(test_docs)\n\n    if strategy == "zero-shot":\n        tagger = ZeroShotPrototypeTagger(\n            model_name=model_name,\n            device=device,\n            zero_shot_margin=zero_shot_margin,\n            zero_shot_threshold=zero_shot_threshold,\n            local_files_only=local_files_only,\n        )\n        pred_map = run_zero_shot(entity_type, test_sentences, tagger)\n        train_docs_used = 0\n    elif strategy == "few-shot":\n        sampled_docs = sample_few_shot_docs(train_docs, few_shot_docs, seed)\n        train_sentences = docs_to_sentences(sampled_docs)\n        train_split, dev_split = split_train_dev(train_sentences, dev_ratio=0.1, seed=seed)\n        if not dev_split and train_split:\n            dev_split = train_split[: min(16, len(train_split))]\n        label_list = build_label_list(entity_type)\n        tokenizer, model, _, id_to_label = train_few_shot_model(\n            model_name=model_name,\n            train_sentences=train_split,\n            dev_sentences=dev_split,\n            label_list=label_list,\n            output_dir=output_dir / f"{entity_type}_few_shot_checkpoints",\n            epochs=epochs,\n            batch_size=batch_size,\n            learning_rate=learning_rate,\n            seed=seed,\n            local_files_only=local_files_only,\n        )\n        pred_map = predict_few_shot_tags(\n            tokenizer=tokenizer,\n            model=model,\n            sentence_examples=test_sentences,\n            id_to_label=id_to_label,\n            device=device,\n        )\n        train_docs_used = len(sampled_docs)\n    else:\n        raise ValueError(f"Unsupported strategy: {strategy}")\n\n    metrics = evaluate_predictions(test_sentences, pred_map)\n    metrics.update(\n        {\n            "entity_type": entity_type,\n            "strategy": strategy,\n            "model_name": model_name,\n            "train_docs_available": len(train_docs),\n            "train_docs_used": train_docs_used,\n            "test_docs_used": len(test_docs),\n        }\n    )\n\n    metrics_path = output_dir / f"{entity_type}_{strategy}_metrics.json"\n    predictions_path = output_dir / f"{entity_type}_{strategy}_predictions.jsonl"\n    output_dir.mkdir(parents=True, exist_ok=True)\n    metrics_path.write_text(\n        json.dumps(to_jsonable(metrics), indent=2),\n        encoding="utf-8",\n    )\n    save_predictions(predictions_path, test_sentences, pred_map)\n    return metrics\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Fine-grained token-level entity extraction with Bio_ClinicalBERT."\n    )\n    parser.add_argument("--data-dir", type=str, default=None)\n    parser.add_argument(\n        "--entity-types",\n        nargs="+",\n        choices=ENTITY_TYPES,\n        default=list(ENTITY_TYPES),\n        help="Which label family to run.",\n    )\n    parser.add_argument(\n        "--strategy",\n        choices=("zero-shot", "few-shot", "both"),\n        default="both",\n        help="Run zero-shot, few-shot, or both.",\n    )\n    parser.add_argument("--model-name", type=str, default=MODEL_NAME)\n    parser.add_argument("--few-shot-docs", type=int, default=40)\n    parser.add_argument("--epochs", type=int, default=3)\n    parser.add_argument("--batch-size", type=int, default=8)\n    parser.add_argument("--learning-rate", type=float, default=2e-5)\n    parser.add_argument("--seed", type=int, default=13)\n    parser.add_argument(\n        "--output-dir",\n        type=Path,\n        default=Path("results") / "clinicalbert_token_level",\n    )\n    parser.add_argument("--device", type=str, default=None)\n    parser.add_argument(\n        "--allow-download",\n        action="store_true",\n        help="Allow Hugging Face downloads instead of using only local cached files.",\n    )\n    parser.add_argument(\n        "--zero-shot-margin",\n        type=float,\n        default=0.03,\n        help="Minimum score margin over the O prototype for a non-O zero-shot prediction.",\n    )\n    parser.add_argument(\n        "--zero-shot-threshold",\n        type=float,\n        default=0.0,\n        help="Minimum cosine score for a non-O zero-shot prediction.",\n    )\n    parser.add_argument(\n        "--max-train-docs",\n        type=int,\n        default=None,\n        help="Optional debug limit for train documents.",\n    )\n    parser.add_argument(\n        "--max-eval-docs",\n        type=int,\n        default=None,\n        help="Optional debug limit for test documents.",\n    )\n    return parser.parse_args()\n\n\ndef main() -> None:\n    args = parse_args()\n    set_seed(args.seed)\n\n    data_dir = get_data_dir(args.data_dir)\n    device = get_device(args.device)\n\n    strategies = ["zero-shot", "few-shot"] if args.strategy == "both" else [args.strategy]\n    all_metrics: List[Dict[str, object]] = []\n\n    for entity_type in args.entity_types:\n        for strategy in strategies:\n            print(\n                f"Running {entity_type:13s} | {strategy:9s} | model={args.model_name}",\n                flush=True,\n            )\n            metrics = run_entity_type(\n                data_dir=data_dir,\n                entity_type=entity_type,\n                strategy=strategy,\n                model_name=args.model_name,\n                output_dir=args.output_dir,\n                few_shot_docs=args.few_shot_docs,\n                epochs=args.epochs,\n                batch_size=args.batch_size,\n                learning_rate=args.learning_rate,\n                seed=args.seed,\n                device=device,\n                zero_shot_margin=args.zero_shot_margin,\n                zero_shot_threshold=args.zero_shot_threshold,\n                local_files_only=not args.allow_download,\n                max_train_docs=args.max_train_docs,\n                max_eval_docs=args.max_eval_docs,\n            )\n            all_metrics.append(metrics)\n            print(\n                json.dumps(\n                    {\n                        "entity_type": metrics["entity_type"],\n                        "strategy": metrics["strategy"],\n                        "precision": round(float(metrics["precision"]), 4),\n                        "recall": round(float(metrics["recall"]), 4),\n                        "f1": round(float(metrics["f1"]), 4),\n                        "token_accuracy": round(float(metrics["token_accuracy"]), 4),\n                        "train_docs_used": metrics["train_docs_used"],\n                        "test_docs_used": metrics["test_docs_used"],\n                    },\n                    indent=2,\n                ),\n                flush=True,\n            )\n\n    summary_path = args.output_dir / "summary.json"\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    summary_path.write_text(\n        json.dumps(to_jsonable(all_metrics), indent=2),\n        encoding="utf-8",\n    )\n    print(f"Saved summary to {summary_path}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'
DP_SOURCE = '#!/usr/bin/env python3\n"""\nDecomposed P/I/O extraction with Bio_ClinicalBERT.\n\nPipeline:\n1. Sentence-level binary filtering for each entity type separately.\n2. Fine-grained token extraction on kept sentences only.\n3. Merge predictions back to document level.\n4. Build a PICO-style table with comparator recovery.\n\nThis module is designed to power a clean notebook deliverable.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom collections import Counter, defaultdict\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Mapping, Sequence, Tuple\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom datasets import Dataset\nfrom sklearn.metrics import (\n    accuracy_score,\n    confusion_matrix,\n    precision_recall_fscore_support,\n)\nfrom transformers import (\n    AutoModel,\n    AutoModelForSequenceClassification,\n    AutoTokenizer,\n    DataCollatorWithPadding,\n    Trainer,\n    TrainingArguments,\n    modeling_utils as transformers_modeling_utils,\n)\n\n\n\ntry:\n    import transformers.safetensors_conversion as safetensors_conversion\n\n    def _disable_auto_conversion(*args, **kwargs):\n        return None, None, None\n\n    safetensors_conversion.auto_conversion = _disable_auto_conversion\n    transformers_modeling_utils.auto_conversion = _disable_auto_conversion\nexcept Exception:\n    pass\n\n\nMODEL_NAME = cte.MODEL_NAME\nENTITY_TYPES = cte.ENTITY_TYPES\nSENT_END = cte.SENT_END\n\nSENTENCE_ZERO_SHOT_PROMPTS: Dict[str, Dict[str, List[str]]] = {\n    "participants": {\n        "positive": [\n            "sentence describing participant demographics",\n            "sentence about patient age sex sample size or condition",\n            "sentence containing participant characteristics or diagnosis",\n        ],\n        "negative": [\n            "sentence not describing participant characteristics",\n            "background or treatment sentence without participant details",\n        ],\n    },\n    "interventions": {\n        "positive": [\n            "sentence describing an intervention treatment or control arm",\n            "sentence about medication surgery therapy education or placebo",\n            "sentence containing how the study treatment was given",\n        ],\n        "negative": [\n            "sentence not describing the intervention",\n            "background or results sentence without treatment details",\n        ],\n    },\n    "outcomes": {\n        "positive": [\n            "sentence describing study outcomes or endpoints",\n            "sentence about physical pain mortality adverse effects or mental outcomes",\n            "sentence containing results measurements or clinical endpoints",\n        ],\n        "negative": [\n            "sentence not describing study outcomes",\n            "background or treatment sentence without outcome details",\n        ],\n    },\n}\n\nCOMPARATOR_KEYWORDS = (\n    "placebo",\n    "control",\n    "usual care",\n    "standard care",\n    "comparator",\n    "sham",\n    "wait list",\n    "waiting list",\n)\n\n\n@dataclass\nclass BinarySentenceExample:\n    doc_id: str\n    sent_idx: int\n    entity_type: str\n    tokens: List[str]\n    tags: List[str]\n    orig_indices: List[int]\n    is_positive: int\n\n    @property\n    def text(self) -> str:\n        return " ".join(self.tokens)\n\n\n@dataclass\nclass DecomposedExperimentArtifacts:\n    entity_type: str\n    strategy: str\n    train_docs: List[cte.Document]\n    test_docs: List[cte.Document]\n    train_sentences: List[BinarySentenceExample]\n    test_sentences: List[BinarySentenceExample]\n    sentence_keep_pred: Dict[Tuple[str, int], int]\n    sentence_keep_gold: Dict[Tuple[str, int], int]\n    sentence_scores: Dict[Tuple[str, int], float]\n    token_sentence_pred_map: Dict[Tuple[str, int], List[str]]\n    token_oracle_sentence_pred_map: Dict[Tuple[str, int], List[str]]\n    doc_tag_map: Dict[str, List[str]]\n    metrics: Dict[str, object]\n\n\ndef get_data_dir(cli_value: str | None = None) -> Path:\n    return cte.get_data_dir(cli_value)\n\n\ndef annotation_inventory(data_dir: Path) -> pd.DataFrame:\n    rows = []\n    roots = [\n        data_dir / "annotations" / "aggregated" / "hierarchical_labels",\n        data_dir / "annotations" / "aggregated" / "starting_spans",\n        data_dir / "annotations" / "individual" / "phase_1",\n        data_dir / "annotations" / "individual" / "phase_2",\n    ]\n    for root in roots:\n        root_name = str(root.relative_to(data_dir))\n        for entity_type in ENTITY_TYPES:\n            entity_root = root / entity_type\n            if not entity_root.exists():\n                continue\n            for split_path in sorted(p for p in entity_root.rglob("*") if p.is_dir()):\n                files = list(split_path.glob("*.ann")) + list(split_path.glob("*.AGGREGATED.ann"))\n                if files:\n                    rows.append(\n                        {\n                            "source": root_name,\n                            "entity_type": entity_type,\n                            "split": str(split_path.relative_to(entity_root)),\n                            "files": len(files),\n                        }\n                    )\n    return pd.DataFrame(rows).sort_values(["source", "entity_type", "split"]).reset_index(drop=True)\n\n\ndef docs_to_binary_sentence_examples(\n    documents: Sequence[cte.Document],\n    entity_type: str,\n    max_tokens: int = 200,\n) -> List[BinarySentenceExample]:\n    examples: List[BinarySentenceExample] = []\n    for doc in documents:\n        chunks = cte.split_into_sentences(doc.tokens, doc.tags, max_tokens=max_tokens)\n        for sent_idx, (tokens, tags, indices) in enumerate(chunks):\n            examples.append(\n                BinarySentenceExample(\n                    doc_id=doc.doc_id,\n                    sent_idx=sent_idx,\n                    entity_type=entity_type,\n                    tokens=tokens,\n                    tags=tags,\n                    orig_indices=indices,\n                    is_positive=int(any(tag != "O" for tag in tags)),\n                )\n            )\n    return examples\n\n\ndef sentence_examples_to_dataset(examples: Sequence[BinarySentenceExample]) -> Dataset:\n    return Dataset.from_dict(\n        {\n            "doc_id": [ex.doc_id for ex in examples],\n            "sent_idx": [ex.sent_idx for ex in examples],\n            "text": [ex.text for ex in examples],\n            "labels": [int(ex.is_positive) for ex in examples],\n        }\n    )\n\n\ndef build_sentence_tokenizer_map(tokenizer):\n    def tokenize_batch(batch):\n        return tokenizer(batch["text"], truncation=True, max_length=256)\n\n    return tokenize_batch\n\n\ndef train_sentence_few_shot_model(\n    model_name: str,\n    train_examples: Sequence[BinarySentenceExample],\n    dev_examples: Sequence[BinarySentenceExample],\n    output_dir: Path,\n    epochs: int,\n    batch_size: int,\n    learning_rate: float,\n    seed: int,\n    local_files_only: bool,\n):\n    tokenizer = AutoTokenizer.from_pretrained(\n        model_name,\n        local_files_only=local_files_only,\n    )\n    train_ds = sentence_examples_to_dataset(train_examples)\n    dev_ds = sentence_examples_to_dataset(dev_examples) if dev_examples else None\n    mapper = build_sentence_tokenizer_map(tokenizer)\n\n    train_tok = train_ds.map(\n        mapper,\n        batched=True,\n        remove_columns=["doc_id", "sent_idx", "text"],\n    )\n    dev_tok = (\n        dev_ds.map(\n            mapper,\n            batched=True,\n            remove_columns=["doc_id", "sent_idx", "text"],\n        )\n        if dev_ds is not None\n        else None\n    )\n\n    model = AutoModelForSequenceClassification.from_pretrained(\n        model_name,\n        num_labels=2,\n        id2label={0: "NEGATIVE", 1: "POSITIVE"},\n        label2id={"NEGATIVE": 0, "POSITIVE": 1},\n        local_files_only=local_files_only,\n        use_safetensors=False,\n    )\n\n    args = TrainingArguments(\n        output_dir=str(output_dir),\n        learning_rate=learning_rate,\n        per_device_train_batch_size=batch_size,\n        per_device_eval_batch_size=batch_size,\n        num_train_epochs=epochs,\n        weight_decay=0.01,\n        optim="adamw_torch",\n        eval_strategy="epoch" if dev_tok is not None else "no",\n        save_strategy="no",\n        logging_strategy="epoch",\n        report_to="none",\n        seed=seed,\n        data_seed=seed,\n        fp16=torch.cuda.is_available(),\n    )\n\n    trainer = Trainer(\n        model=model,\n        args=args,\n        train_dataset=train_tok,\n        eval_dataset=dev_tok,\n        processing_class=tokenizer,\n        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),\n    )\n    trainer.train()\n    return tokenizer, model\n\n\n@torch.no_grad()\ndef predict_sentence_few_shot(\n    tokenizer,\n    model,\n    examples: Sequence[BinarySentenceExample],\n    device: torch.device,\n) -> Tuple[Dict[Tuple[str, int], int], Dict[Tuple[str, int], float]]:\n    model.to(device)\n    model.eval()\n\n    keep_map: Dict[Tuple[str, int], int] = {}\n    score_map: Dict[Tuple[str, int], float] = {}\n\n    for example in examples:\n        encoding = tokenizer(\n            example.text,\n            return_tensors="pt",\n            truncation=True,\n            max_length=256,\n        )\n        inputs = {key: value.to(device) for key, value in encoding.items()}\n        logits = model(**inputs).logits[0]\n        probs = torch.softmax(logits, dim=-1)\n        positive_score = float(probs[1].detach().cpu())\n        key = (example.doc_id, example.sent_idx)\n        keep_map[key] = int(positive_score >= 0.5)\n        score_map[key] = positive_score\n\n    return keep_map, score_map\n\n\nclass BioClinicalSentenceEmbedder:\n    def __init__(self, model_name: str, device: torch.device, local_files_only: bool) -> None:\n        self.device = device\n        self.tokenizer = AutoTokenizer.from_pretrained(\n            model_name,\n            local_files_only=local_files_only,\n        )\n        self.model = AutoModel.from_pretrained(\n            model_name,\n            local_files_only=local_files_only,\n            use_safetensors=False,\n        ).to(device)\n        self.model.eval()\n\n    @torch.no_grad()\n    def encode(self, texts: Sequence[str], batch_size: int = 8, max_length: int = 256) -> torch.Tensor:\n        outputs: List[torch.Tensor] = []\n        for start in range(0, len(texts), batch_size):\n            batch = list(texts[start : start + batch_size])\n            encoding = self.tokenizer(\n                batch,\n                return_tensors="pt",\n                padding=True,\n                truncation=True,\n                max_length=max_length,\n                return_special_tokens_mask=True,\n            )\n            special_tokens_mask = encoding.pop("special_tokens_mask").to(self.device)\n            inputs = {key: value.to(self.device) for key, value in encoding.items()}\n            model_outputs = self.model(**inputs, output_hidden_states=True)\n            hidden = torch.stack(model_outputs.hidden_states[-4:], dim=0).mean(dim=0)\n\n            for row_hidden, row_mask in zip(hidden, special_tokens_mask):\n                usable = row_hidden[~row_mask.bool()]\n                if usable.numel() == 0:\n                    usable = row_hidden\n                pooled = F.normalize(usable.mean(dim=0), dim=0)\n                outputs.append(pooled.detach().cpu())\n        return torch.stack(outputs, dim=0) if outputs else torch.empty(0, 768)\n\n\nclass SentenceZeroShotClassifier:\n    def __init__(\n        self,\n        model_name: str,\n        device: torch.device,\n        local_files_only: bool,\n        margin: float = 0.02,\n    ) -> None:\n        self.embedder = BioClinicalSentenceEmbedder(\n            model_name=model_name,\n            device=device,\n            local_files_only=local_files_only,\n        )\n        self.margin = margin\n        self.cache: Dict[str, Tuple[torch.Tensor, torch.Tensor]] = {}\n\n    def _build_prototypes(self, entity_type: str) -> Tuple[torch.Tensor, torch.Tensor]:\n        if entity_type not in self.cache:\n            prompt_map = SENTENCE_ZERO_SHOT_PROMPTS[entity_type]\n            positive = self.embedder.encode(prompt_map["positive"]).mean(dim=0)\n            negative = self.embedder.encode(prompt_map["negative"]).mean(dim=0)\n            positive = F.normalize(positive, dim=0)\n            negative = F.normalize(negative, dim=0)\n            self.cache[entity_type] = (positive, negative)\n        return self.cache[entity_type]\n\n    def predict(\n        self,\n        entity_type: str,\n        examples: Sequence[BinarySentenceExample],\n    ) -> Tuple[Dict[Tuple[str, int], int], Dict[Tuple[str, int], float]]:\n        positive, negative = self._build_prototypes(entity_type)\n        text_embeddings = self.embedder.encode([ex.text for ex in examples])\n        keep_map: Dict[Tuple[str, int], int] = {}\n        score_map: Dict[Tuple[str, int], float] = {}\n\n        for example, emb in zip(examples, text_embeddings):\n            emb = F.normalize(emb, dim=0)\n            pos_score = float(torch.dot(emb, positive))\n            neg_score = float(torch.dot(emb, negative))\n            logit_gap = pos_score - neg_score\n            probability = float(torch.softmax(torch.tensor([neg_score, pos_score]), dim=0)[1])\n            key = (example.doc_id, example.sent_idx)\n            keep_map[key] = int(logit_gap > self.margin)\n            score_map[key] = probability\n\n        return keep_map, score_map\n\n\ndef evaluate_sentence_predictions(\n    examples: Sequence[BinarySentenceExample],\n    keep_map: Mapping[Tuple[str, int], int],\n    score_map: Mapping[Tuple[str, int], float] | None = None,\n) -> Dict[str, object]:\n    gold = [int(ex.is_positive) for ex in examples]\n    pred = [int(keep_map[(ex.doc_id, ex.sent_idx)]) for ex in examples]\n\n    precision, recall, f1, _ = precision_recall_fscore_support(\n        gold,\n        pred,\n        average="binary",\n        zero_division=0,\n    )\n    cm = confusion_matrix(gold, pred, labels=[0, 1]).tolist()\n    metrics = {\n        "accuracy": float(accuracy_score(gold, pred)),\n        "precision": float(precision),\n        "recall": float(recall),\n        "f1": float(f1),\n        "keep_rate": float(np.mean(pred)) if pred else 0.0,\n        "confusion_matrix": cm,\n    }\n    if score_map:\n        metrics["mean_positive_score"] = float(\n            np.mean([score_map[(ex.doc_id, ex.sent_idx)] for ex in examples])\n        )\n    return metrics\n\n\ndef sentence_keep_map_from_gold(examples: Sequence[BinarySentenceExample]) -> Dict[Tuple[str, int], int]:\n    return {(ex.doc_id, ex.sent_idx): int(ex.is_positive) for ex in examples}\n\n\ndef filter_examples_by_keep_map(\n    examples: Sequence[BinarySentenceExample],\n    keep_map: Mapping[Tuple[str, int], int],\n) -> List[BinarySentenceExample]:\n    return [ex for ex in examples if int(keep_map[(ex.doc_id, ex.sent_idx)]) == 1]\n\n\ndef token_predictions_for_kept_sentences(\n    strategy: str,\n    entity_type: str,\n    train_examples: Sequence[BinarySentenceExample],\n    test_examples: Sequence[BinarySentenceExample],\n    kept_test_examples: Sequence[BinarySentenceExample],\n    output_dir: Path,\n    few_shot_docs: int,\n    epochs: int,\n    batch_size: int,\n    learning_rate: float,\n    seed: int,\n    device: torch.device,\n    local_files_only: bool,\n) -> Dict[Tuple[str, int], List[str]]:\n    if strategy == "zero-shot":\n        tagger = cte.ZeroShotPrototypeTagger(\n            model_name=MODEL_NAME,\n            device=device,\n            zero_shot_margin=0.03,\n            zero_shot_threshold=0.0,\n            local_files_only=local_files_only,\n        )\n        return cte.run_zero_shot(entity_type, kept_test_examples, tagger)\n\n    positive_train_examples = [ex for ex in train_examples if ex.is_positive == 1]\n    train_split, dev_split = cte.split_train_dev(positive_train_examples, dev_ratio=0.1, seed=seed)\n    if not dev_split and train_split:\n        dev_split = train_split[: min(16, len(train_split))]\n    label_list = cte.build_label_list(entity_type)\n\n    tokenizer, model, _, id_to_label = cte.train_few_shot_model(\n        model_name=MODEL_NAME,\n        train_sentences=train_split,\n        dev_sentences=dev_split,\n        label_list=label_list,\n        output_dir=output_dir / f"{entity_type}_token_few_shot_checkpoints",\n        epochs=epochs,\n        batch_size=batch_size,\n        learning_rate=learning_rate,\n        seed=seed,\n        local_files_only=local_files_only,\n    )\n    return cte.predict_few_shot_tags(\n        tokenizer=tokenizer,\n        model=model,\n        sentence_examples=kept_test_examples,\n        id_to_label=id_to_label,\n        device=device,\n    )\n\n\ndef expand_sentence_tag_map(\n    examples: Sequence[BinarySentenceExample],\n    keep_map: Mapping[Tuple[str, int], int],\n    positive_pred_map: Mapping[Tuple[str, int], Sequence[str]],\n) -> Dict[Tuple[str, int], List[str]]:\n    full_map: Dict[Tuple[str, int], List[str]] = {}\n    for example in examples:\n        key = (example.doc_id, example.sent_idx)\n        if int(keep_map[key]) == 1:\n            tags = list(positive_pred_map.get(key, ["O"] * len(example.tokens)))\n            if len(tags) < len(example.tokens):\n                tags.extend(["O"] * (len(example.tokens) - len(tags)))\n            full_map[key] = tags[: len(example.tokens)]\n        else:\n            full_map[key] = ["O"] * len(example.tokens)\n    return full_map\n\n\ndef merge_sentence_predictions_to_docs(\n    test_docs: Sequence[cte.Document],\n    test_examples: Sequence[BinarySentenceExample],\n    sentence_tag_map: Mapping[Tuple[str, int], Sequence[str]],\n) -> Dict[str, List[str]]:\n    merged = {doc.doc_id: ["O"] * len(doc.tokens) for doc in test_docs}\n    for example in test_examples:\n        tags = list(sentence_tag_map[(example.doc_id, example.sent_idx)])\n        for orig_idx, tag in zip(example.orig_indices, tags):\n            merged[example.doc_id][orig_idx] = tag\n    return merged\n\n\ndef spans_from_typed_bio(tags: Sequence[str]) -> List[Tuple[int, int, str]]:\n    spans: List[Tuple[int, int, str]] = []\n    start = None\n    current_label = None\n\n    for idx, tag in enumerate(tags):\n        if tag == "O":\n            if start is not None and current_label is not None:\n                spans.append((start, idx - 1, current_label))\n                start = None\n                current_label = None\n            continue\n\n        prefix, label_name = tag.split("-", 1)\n        if prefix == "B" or current_label != label_name:\n            if start is not None and current_label is not None:\n                spans.append((start, idx - 1, current_label))\n            start = idx\n            current_label = label_name\n        elif prefix == "I" and current_label == label_name:\n            continue\n\n    if start is not None and current_label is not None:\n        spans.append((start, len(tags) - 1, current_label))\n    return spans\n\n\ndef normalize_span_text(text: str) -> str:\n    return " ".join(text.replace(" ,", ",").replace(" .", ".").split())\n\n\ndef span_texts_by_label(tokens: Sequence[str], tags: Sequence[str]) -> Dict[str, List[str]]:\n    output: Dict[str, List[str]] = defaultdict(list)\n    for start, end, label_name in spans_from_typed_bio(tags):\n        output[label_name].append(normalize_span_text(" ".join(tokens[start : end + 1])))\n    return dict(output)\n\n\ndef unique_join(items: Iterable[str]) -> str:\n    deduped: List[str] = []\n    seen = set()\n    for item in items:\n        cleaned = normalize_span_text(item)\n        if cleaned and cleaned not in seen:\n            deduped.append(cleaned)\n            seen.add(cleaned)\n    return " | ".join(deduped)\n\n\ndef recover_comparator(tokens: Sequence[str], intervention_map: Mapping[str, List[str]]) -> str:\n    if "CONTROL" in intervention_map and intervention_map["CONTROL"]:\n        return unique_join(intervention_map["CONTROL"])\n\n    full_text = " ".join(tokens).lower()\n    matches = [kw for kw in COMPARATOR_KEYWORDS if kw in full_text]\n    if matches:\n        return unique_join(matches)\n    return ""\n\n\ndef build_pico_table(\n    common_doc_ids: Sequence[str],\n    participants_docs: Mapping[str, cte.Document],\n    interventions_docs: Mapping[str, cte.Document],\n    outcomes_docs: Mapping[str, cte.Document],\n    participants_pred: Mapping[str, Sequence[str]],\n    interventions_pred: Mapping[str, Sequence[str]],\n    outcomes_pred: Mapping[str, Sequence[str]],\n) -> pd.DataFrame:\n    rows = []\n    for doc_id in common_doc_ids:\n        p_doc = participants_docs[doc_id]\n        i_doc = interventions_docs[doc_id]\n        o_doc = outcomes_docs[doc_id]\n\n        p_map = span_texts_by_label(p_doc.tokens, participants_pred[doc_id])\n        i_map = span_texts_by_label(i_doc.tokens, interventions_pred[doc_id])\n        o_map = span_texts_by_label(o_doc.tokens, outcomes_pred[doc_id])\n\n        active_interventions = []\n        for key, values in i_map.items():\n            if key != "CONTROL":\n                active_interventions.extend(values)\n\n        all_outcomes = []\n        for values in o_map.values():\n            all_outcomes.extend(values)\n\n        rows.append(\n            {\n                "doc_id": doc_id,\n                "population": unique_join(\n                    p_map.get("CONDITION", [])\n                    + p_map.get("AGE", [])\n                    + p_map.get("SEX", [])\n                    + p_map.get("SAMPLE_SIZE", [])\n                ),\n                "participant_age": unique_join(p_map.get("AGE", [])),\n                "participant_sex": unique_join(p_map.get("SEX", [])),\n                "participant_sample_size": unique_join(p_map.get("SAMPLE_SIZE", [])),\n                "participant_condition": unique_join(p_map.get("CONDITION", [])),\n                "intervention": unique_join(active_interventions),\n                "comparator": recover_comparator(i_doc.tokens, i_map),\n                "outcome": unique_join(all_outcomes),\n                "outcome_physical": unique_join(o_map.get("PHYSICAL", [])),\n                "outcome_pain": unique_join(o_map.get("PAIN", [])),\n                "outcome_mortality": unique_join(o_map.get("MORTALITY", [])),\n                "outcome_adverse_effects": unique_join(o_map.get("ADVERSE_EFFECTS", [])),\n                "outcome_mental": unique_join(o_map.get("MENTAL", [])),\n                "outcome_other": unique_join(o_map.get("OTHER", [])),\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef save_json(path: Path, payload) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(cte.to_jsonable(payload), indent=2), encoding="utf-8")\n\n\ndef save_jsonl(path: Path, rows: Sequence[Mapping[str, object]]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as handle:\n        for row in rows:\n            handle.write(json.dumps(cte.to_jsonable(row)) + "\\n")\n\n\ndef run_decomposed_experiment(\n    data_dir: Path,\n    entity_type: str,\n    strategy: str,\n    output_dir: Path,\n    few_shot_docs: int = 40,\n    epochs: int = 3,\n    batch_size: int = 8,\n    learning_rate: float = 2e-5,\n    seed: int = 13,\n    device: torch.device | None = None,\n    local_files_only: bool = True,\n    max_train_docs: int | None = None,\n    max_eval_docs: int | None = None,\n) -> DecomposedExperimentArtifacts:\n    if device is None:\n        device = cte.get_device(None)\n\n    train_docs = cte.load_dataset(data_dir, entity_type, "train", limit=max_train_docs)\n    test_docs = cte.load_dataset(data_dir, entity_type, "test", limit=max_eval_docs)\n\n    if strategy == "few-shot":\n        train_docs = cte.sample_few_shot_docs(train_docs, few_shot_docs, seed)\n\n    train_sentences = docs_to_binary_sentence_examples(train_docs, entity_type)\n    test_sentences = docs_to_binary_sentence_examples(test_docs, entity_type)\n    gold_keep_map = sentence_keep_map_from_gold(test_sentences)\n\n    if strategy == "zero-shot":\n        sentence_classifier = SentenceZeroShotClassifier(\n            model_name=MODEL_NAME,\n            device=device,\n            local_files_only=local_files_only,\n        )\n        sentence_keep_pred, sentence_scores = sentence_classifier.predict(entity_type, test_sentences)\n    elif strategy == "few-shot":\n        train_split, dev_split = cte.split_train_dev(train_sentences, dev_ratio=0.1, seed=seed)\n        if not dev_split and train_split:\n            dev_split = train_split[: min(16, len(train_split))]\n        sent_tokenizer, sent_model = train_sentence_few_shot_model(\n            model_name=MODEL_NAME,\n            train_examples=train_split,\n            dev_examples=dev_split,\n            output_dir=output_dir / f"{entity_type}_sentence_few_shot_checkpoints",\n            epochs=epochs,\n            batch_size=batch_size,\n            learning_rate=learning_rate,\n            seed=seed,\n            local_files_only=local_files_only,\n        )\n        sentence_keep_pred, sentence_scores = predict_sentence_few_shot(\n            tokenizer=sent_tokenizer,\n            model=sent_model,\n            examples=test_sentences,\n            device=device,\n        )\n    else:\n        raise ValueError(f"Unsupported strategy: {strategy}")\n\n    kept_test_pred = filter_examples_by_keep_map(test_sentences, sentence_keep_pred)\n    kept_test_gold = filter_examples_by_keep_map(test_sentences, gold_keep_map)\n\n    if strategy == "zero-shot":\n        token_tagger = cte.ZeroShotPrototypeTagger(\n            model_name=MODEL_NAME,\n            device=device,\n            zero_shot_margin=0.03,\n            zero_shot_threshold=0.0,\n            local_files_only=local_files_only,\n        )\n        token_sentence_pred_positive_map = cte.run_zero_shot(\n            entity_type,\n            kept_test_pred,\n            token_tagger,\n        )\n        token_sentence_oracle_positive_map = cte.run_zero_shot(\n            entity_type,\n            kept_test_gold,\n            token_tagger,\n        )\n    else:\n        positive_train_examples = [ex for ex in train_sentences if ex.is_positive == 1]\n        if not positive_train_examples:\n            positive_train_examples = train_sentences[: min(16, len(train_sentences))]\n        token_train_split, token_dev_split = cte.split_train_dev(\n            positive_train_examples,\n            dev_ratio=0.1,\n            seed=seed,\n        )\n        if not token_dev_split and token_train_split:\n            token_dev_split = token_train_split[: min(16, len(token_train_split))]\n\n        label_list = cte.build_label_list(entity_type)\n        token_tokenizer, token_model, _, token_id_to_label = cte.train_few_shot_model(\n            model_name=MODEL_NAME,\n            train_sentences=token_train_split,\n            dev_sentences=token_dev_split,\n            label_list=label_list,\n            output_dir=output_dir / f"{entity_type}_token_few_shot_checkpoints",\n            epochs=epochs,\n            batch_size=batch_size,\n            learning_rate=learning_rate,\n            seed=seed,\n            local_files_only=local_files_only,\n        )\n        token_sentence_pred_positive_map = cte.predict_few_shot_tags(\n            tokenizer=token_tokenizer,\n            model=token_model,\n            sentence_examples=kept_test_pred,\n            id_to_label=token_id_to_label,\n            device=device,\n        )\n        token_sentence_oracle_positive_map = cte.predict_few_shot_tags(\n            tokenizer=token_tokenizer,\n            model=token_model,\n            sentence_examples=kept_test_gold,\n            id_to_label=token_id_to_label,\n            device=device,\n        )\n\n    pipeline_sentence_tag_map = expand_sentence_tag_map(\n        test_sentences,\n        sentence_keep_pred,\n        token_sentence_pred_positive_map,\n    )\n    oracle_sentence_tag_map = expand_sentence_tag_map(\n        test_sentences,\n        gold_keep_map,\n        token_sentence_oracle_positive_map,\n    )\n\n    sentence_metrics = evaluate_sentence_predictions(\n        test_sentences,\n        sentence_keep_pred,\n        sentence_scores,\n    )\n    token_pipeline_metrics = cte.evaluate_predictions(test_sentences, pipeline_sentence_tag_map)\n    token_oracle_metrics = cte.evaluate_predictions(test_sentences, oracle_sentence_tag_map)\n\n    doc_tag_map = merge_sentence_predictions_to_docs(\n        test_docs=test_docs,\n        test_examples=test_sentences,\n        sentence_tag_map=pipeline_sentence_tag_map,\n    )\n\n    metrics = {\n        "entity_type": entity_type,\n        "strategy": strategy,\n        "train_docs_used": len(train_docs),\n        "test_docs_used": len(test_docs),\n        "sentence_accuracy": sentence_metrics["accuracy"],\n        "sentence_precision": sentence_metrics["precision"],\n        "sentence_recall": sentence_metrics["recall"],\n        "sentence_f1": sentence_metrics["f1"],\n        "sentence_keep_rate": sentence_metrics["keep_rate"],\n        "token_pipeline_precision": token_pipeline_metrics["precision"],\n        "token_pipeline_recall": token_pipeline_metrics["recall"],\n        "token_pipeline_f1": token_pipeline_metrics["f1"],\n        "token_pipeline_token_accuracy": token_pipeline_metrics["token_accuracy"],\n        "token_oracle_precision": token_oracle_metrics["precision"],\n        "token_oracle_recall": token_oracle_metrics["recall"],\n        "token_oracle_f1": token_oracle_metrics["f1"],\n        "token_oracle_token_accuracy": token_oracle_metrics["token_accuracy"],\n        "token_filter_loss": token_oracle_metrics["f1"] - token_pipeline_metrics["f1"],\n        "sentence_metrics": sentence_metrics,\n        "token_pipeline_metrics": token_pipeline_metrics,\n        "token_oracle_metrics": token_oracle_metrics,\n    }\n\n    save_json(output_dir / f"{entity_type}_{strategy}_metrics.json", metrics)\n    save_jsonl(\n        output_dir / f"{entity_type}_{strategy}_sentence_predictions.jsonl",\n        [\n            {\n                "doc_id": ex.doc_id,\n                "sent_idx": ex.sent_idx,\n                "text": ex.text,\n                "gold_keep": ex.is_positive,\n                "pred_keep": sentence_keep_pred[(ex.doc_id, ex.sent_idx)],\n                "score": sentence_scores[(ex.doc_id, ex.sent_idx)],\n            }\n            for ex in test_sentences\n        ],\n    )\n    save_jsonl(\n        output_dir / f"{entity_type}_{strategy}_token_predictions.jsonl",\n        [\n            {\n                "doc_id": ex.doc_id,\n                "sent_idx": ex.sent_idx,\n                "tokens": ex.tokens,\n                "gold_tags": ex.tags,\n                "pred_tags": pipeline_sentence_tag_map[(ex.doc_id, ex.sent_idx)],\n            }\n            for ex in test_sentences\n        ],\n    )\n\n    return DecomposedExperimentArtifacts(\n        entity_type=entity_type,\n        strategy=strategy,\n        train_docs=train_docs,\n        test_docs=test_docs,\n        train_sentences=train_sentences,\n        test_sentences=test_sentences,\n        sentence_keep_pred=sentence_keep_pred,\n        sentence_keep_gold=gold_keep_map,\n        sentence_scores=sentence_scores,\n        token_sentence_pred_map=pipeline_sentence_tag_map,\n        token_oracle_sentence_pred_map=oracle_sentence_tag_map,\n        doc_tag_map=doc_tag_map,\n        metrics=metrics,\n    )\n\n\ndef run_all_decomposed_experiments(\n    data_dir: Path,\n    entity_types: Sequence[str] = ENTITY_TYPES,\n    strategies: Sequence[str] = ("zero-shot", "few-shot"),\n    output_dir: Path | None = None,\n    few_shot_docs: int = 40,\n    epochs: int = 3,\n    batch_size: int = 8,\n    learning_rate: float = 2e-5,\n    seed: int = 13,\n    device: torch.device | None = None,\n    local_files_only: bool = True,\n    max_train_docs: int | None = None,\n    max_eval_docs: int | None = None,\n) -> Tuple[pd.DataFrame, Dict[Tuple[str, str], DecomposedExperimentArtifacts]]:\n    if output_dir is None:\n        output_dir = data_dir / "results" / "clinicalbert_decomposed_pio"\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    artifacts: Dict[Tuple[str, str], DecomposedExperimentArtifacts] = {}\n    rows: List[Dict[str, object]] = []\n\n    for entity_type in entity_types:\n        for strategy in strategies:\n            artifact = run_decomposed_experiment(\n                data_dir=data_dir,\n                entity_type=entity_type,\n                strategy=strategy,\n                output_dir=output_dir,\n                few_shot_docs=few_shot_docs,\n                epochs=epochs,\n                batch_size=batch_size,\n                learning_rate=learning_rate,\n                seed=seed,\n                device=device,\n                local_files_only=local_files_only,\n                max_train_docs=max_train_docs,\n                max_eval_docs=max_eval_docs,\n            )\n            artifacts[(entity_type, strategy)] = artifact\n            rows.append(\n                {\n                    key: value\n                    for key, value in artifact.metrics.items()\n                    if key\n                    not in ("sentence_metrics", "token_pipeline_metrics", "token_oracle_metrics")\n                }\n            )\n\n    summary_df = pd.DataFrame(rows).sort_values(["entity_type", "strategy"]).reset_index(drop=True)\n    save_json(output_dir / "summary.json", rows)\n    return summary_df, artifacts\n\n\ndef build_strategy_pico_table(\n    artifacts: Mapping[Tuple[str, str], DecomposedExperimentArtifacts],\n    strategy: str,\n) -> pd.DataFrame:\n    p_art = artifacts[("participants", strategy)]\n    i_art = artifacts[("interventions", strategy)]\n    o_art = artifacts[("outcomes", strategy)]\n\n    p_docs = {doc.doc_id: doc for doc in p_art.test_docs}\n    i_docs = {doc.doc_id: doc for doc in i_art.test_docs}\n    o_docs = {doc.doc_id: doc for doc in o_art.test_docs}\n    common_doc_ids = sorted(set(p_docs) & set(i_docs) & set(o_docs))\n\n    return build_pico_table(\n        common_doc_ids=common_doc_ids,\n        participants_docs=p_docs,\n        interventions_docs=i_docs,\n        outcomes_docs=o_docs,\n        participants_pred=p_art.doc_tag_map,\n        interventions_pred=i_art.doc_tag_map,\n        outcomes_pred=o_art.doc_tag_map,\n    )\n'

cte = load_inline_module(
    "clinicalbert_token_level_extraction",
    CTE_SOURCE,
    PROJECT_DIR / "clinicalbert_token_level_extraction.py",
)
dp = load_inline_module(
    "clinicalbert_decomposed_pio",
    DP_SOURCE,
    PROJECT_DIR / "clinicalbert_decomposed_pio.py",
    extra_globals={"cte": cte},
)

DATA_DIR = dp.get_data_dir(None)
OUTPUT_DIR = DATA_DIR / "results" / "decomposed_pico_bioclinicalbert"
MODEL_NAME = dp.MODEL_NAME
SEED = 13

cte.set_seed(SEED)
DEVICE = cte.get_device(None)

print("Project dir:", PROJECT_DIR)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)
print("Model:", MODEL_NAME)
print("Device:", DEVICE)
print("Token module lines:", len(CTE_SOURCE.splitlines()))
print("Decomposed module lines:", len(DP_SOURCE.splitlines()))


### Self-Contained Source Check

This cell confirms that the notebook already contains the working implementation and does not depend on external helper imports at runtime.


In [ ]:
print("clinicalbert_token_level_extraction inline:", cte.__name__)
print("clinicalbert_decomposed_pio inline:", dp.__name__)
print("token source loaded:", hasattr(cte, "run_zero_shot"))
print("decomposed source loaded:", hasattr(dp, "run_all_decomposed_experiments"))


## 2. What Is Inside the Dataset?

EBM-NLP provides:

- raw documents
- aggregated annotations
- individual annotations
- separate material for participants, interventions, and outcomes
- train and test splits
- hierarchical labels
- starting spans

The final decomposed pipeline uses:

- **aggregated hierarchical labels** for training and evaluation
- **gold test annotations** for evaluation

We still inspect the other folders so it is clear what data exists and what we are choosing not to use directly.


In [ ]:
inventory_df = dp.annotation_inventory(DATA_DIR)
display(inventory_df)


### Why Use Aggregated Hierarchical Labels?

We use `annotations/aggregated/hierarchical_labels/...` because:

1. they already combine annotation evidence into a stable target
2. they preserve the subtype IDs we need for token identity extraction
3. they are better suited for professor-facing evaluation than raw individual annotation disagreement

We do **not** train directly on:

- crowd test labels
- individual phase 1 labels
- individual phase 2 labels

Those can be used later for additional analysis, but not for the main final pipeline.


In [ ]:
doc_summary_rows = []
for entity_type in dp.ENTITY_TYPES:
    train_docs = cte.load_dataset(DATA_DIR, entity_type, "train")
    test_docs = cte.load_dataset(DATA_DIR, entity_type, "test")
    doc_summary_rows.append(
        {
            "entity_type": entity_type,
            "train_docs": len(train_docs),
            "test_docs": len(test_docs),
            "train_tokens": sum(len(doc.tokens) for doc in train_docs),
            "test_tokens": sum(len(doc.tokens) for doc in test_docs),
        }
    )

doc_summary_df = pd.DataFrame(doc_summary_rows)
display(doc_summary_df)


## 3. Looking at Raw Documents

Before modeling, we inspect raw tokenized documents.

This is important because the professor should be able to see that:

1. the corpus is already tokenized
2. annotations align token-by-token with the documents
3. P/I/O mentions are distributed across real abstract text, not synthetic examples


In [ ]:
participants_train = cte.load_dataset(DATA_DIR, "participants", "train")
interventions_train = cte.load_dataset(DATA_DIR, "interventions", "train")
outcomes_train = cte.load_dataset(DATA_DIR, "outcomes", "train")

common_train_ids = sorted(
    set(doc.doc_id for doc in participants_train)
    & set(doc.doc_id for doc in interventions_train)
    & set(doc.doc_id for doc in outcomes_train)
)

sample_doc_id = common_train_ids[0]
sample_tokens = cte.load_tokens(DATA_DIR, sample_doc_id)

print("Sample train document ID:", sample_doc_id)
print("First 150 tokens:")
print(" ".join(sample_tokens[:150]))


## 4. Fine-Grained Label Schema

The key improvement in this notebook is that token extraction uses the full subtype schema instead of collapsing everything into plain `B/I/O`.

That means we distinguish:

- `B-AGE` from `B-CONDITION`
- `B-DRUG` from `B-CONTROL`
- `B-MORTALITY` from `B-PAIN`

This is essential for token identity extraction.


In [ ]:
schema_rows = []
for entity_type, mapping in cte.ENTITY_LABELS.items():
    for label_id, label_name in mapping.items():
        schema_rows.append(
            {
                "entity_type": entity_type,
                "label_id": label_id,
                "label_name": label_name,
            }
        )

schema_df = pd.DataFrame(schema_rows)
display(schema_df)


### Typed BIO Tag Space

For each entity type we convert the hierarchical subtype IDs into typed BIO tags.

For example:

- Participants: `O`, `B-AGE`, `I-AGE`, `B-SEX`, `I-SEX`, ...
- Interventions: `O`, `B-DRUG`, `I-DRUG`, `B-CONTROL`, `I-CONTROL`, ...
- Outcomes: `O`, `B-PAIN`, `I-PAIN`, `B-MORTALITY`, `I-MORTALITY`, ...


In [ ]:
typed_bio_rows = []
for entity_type in dp.ENTITY_TYPES:
    typed_bio_rows.append(
        {
            "entity_type": entity_type,
            "typed_bio_tags": ", ".join(cte.build_label_list(entity_type)),
        }
    )

typed_bio_df = pd.DataFrame(typed_bio_rows)
display(typed_bio_df)


## 5. Inspecting Label Alignment

The next step is to prove that the fine-grained labels really line up with the tokenized text.

We show:

1. token
2. hierarchical label ID
3. converted typed BIO tag

for a positive example from each entity type.


In [ ]:
def preview_alignment(entity_type: str, split: str = "train", max_rows: int = 40):
    docs = cte.load_dataset(DATA_DIR, entity_type, split)
    for doc in docs:
        if any(label_id != 0 for label_id in doc.label_ids):
            rows = []
            for idx, (tok, label_id, tag) in enumerate(zip(doc.tokens, doc.label_ids, doc.tags)):
                if label_id != 0:
                    start = max(0, idx - 5)
                    end = min(len(doc.tokens), idx + max_rows)
                    for j in range(start, end):
                        rows.append(
                            {
                                "doc_id": doc.doc_id,
                                "index": j,
                                "token": doc.tokens[j],
                                "label_id": doc.label_ids[j],
                                "typed_bio_tag": doc.tags[j],
                            }
                        )
                    return pd.DataFrame(rows).drop_duplicates(subset=["index"]).reset_index(drop=True)
    return pd.DataFrame()


print("Participants example")
display(preview_alignment("participants"))

print("Interventions example")
display(preview_alignment("interventions"))

print("Outcomes example")
display(preview_alignment("outcomes"))


## 6. Why Not Use `starting_spans` as the Main Token Target?

The dataset also contains `starting_spans`, but for this notebook we derive span boundaries directly from subtype transitions in `hierarchical_labels`.

Reason:

1. hierarchical labels already contain the subtype identity we need
2. switching subtype IDs naturally gives us boundary information
3. this keeps the training target simple and reproducible

So the final token tags are created as:

- `O` when the hierarchical label is `0`
- `B-LABEL` when the current subtype differs from the previous token
- `I-LABEL` when the current subtype continues the same span


## 7. Decomposed Pipeline Overview

The full pipeline is:

1. load documents and fine-grained labels
2. split each document into sentences
3. create one **binary sentence filter** per entity type:
   - Participants vs non-Participants
   - Interventions vs non-Interventions
   - Outcomes vs non-Outcomes
4. keep only the predicted positive sentences for each entity type
5. run fine-grained token extraction on those kept sentences
6. merge sentence predictions back into document-level tag sequences
7. convert the document-level tags into a final PICO table

We also compute an **oracle token score**, where token extraction receives the gold sentence filter. This tells us how much loss comes from the sentence stage.


In [ ]:
pipeline_steps_df = pd.DataFrame(
    [
        {"step": 1, "component": "Data loading", "purpose": "Load tokens and fine-grained hierarchical labels"},
        {"step": 2, "component": "Sentence segmentation", "purpose": "Break documents into sentence units"},
        {"step": 3, "component": "Sentence filter", "purpose": "Keep only relevant sentences for each entity type"},
        {"step": 4, "component": "Token extractor", "purpose": "Predict fine-grained typed BIO labels on kept sentences"},
        {"step": 5, "component": "Merge", "purpose": "Map sentence predictions back to original document token positions"},
        {"step": 6, "component": "PICO construction", "purpose": "Assemble population, intervention, comparator, and outcome columns"},
        {"step": 7, "component": "Evaluation", "purpose": "Measure sentence and token performance and inspect bottlenecks"},
    ]
)
display(pipeline_steps_df)


## 8. Sentence Construction and Positive/Negative Labels

For the decomposed method, every sentence gets a binary label:

- `1` if it contains at least one token of the target entity type
- `0` otherwise

This lets us train a sentence filter for each entity type separately.


In [ ]:
sentence_summary_rows = []
for entity_type in dp.ENTITY_TYPES:
    train_docs = cte.load_dataset(DATA_DIR, entity_type, "train")
    test_docs = cte.load_dataset(DATA_DIR, entity_type, "test")
    train_sentences = dp.docs_to_binary_sentence_examples(train_docs, entity_type)
    test_sentences = dp.docs_to_binary_sentence_examples(test_docs, entity_type)
    sentence_summary_rows.append(
        {
            "entity_type": entity_type,
            "train_sentences": len(train_sentences),
            "test_sentences": len(test_sentences),
            "train_positive": sum(x.is_positive for x in train_sentences),
            "train_negative": len(train_sentences) - sum(x.is_positive for x in train_sentences),
            "test_positive": sum(x.is_positive for x in test_sentences),
            "test_negative": len(test_sentences) - sum(x.is_positive for x in test_sentences),
        }
    )

sentence_summary_df = pd.DataFrame(sentence_summary_rows)
display(sentence_summary_df)


In [ ]:
def preview_sentence_examples(entity_type: str, split: str = "train", n: int = 5):
    docs = cte.load_dataset(DATA_DIR, entity_type, split)
    examples = dp.docs_to_binary_sentence_examples(docs, entity_type)
    rows = []
    for ex in examples[:n]:
        rows.append(
            {
                "doc_id": ex.doc_id,
                "sent_idx": ex.sent_idx,
                "entity_type": ex.entity_type,
                "is_positive": ex.is_positive,
                "sentence": ex.text,
            }
        )
    return pd.DataFrame(rows)


display(preview_sentence_examples("participants"))
display(preview_sentence_examples("interventions"))
display(preview_sentence_examples("outcomes"))


## 9. Zero-Shot and Few-Shot Strategy

We evaluate both stages in two modes.

### Zero-shot

- sentence stage: prototype similarity using Bio_ClinicalBERT sentence embeddings
- token stage: prototype similarity using Bio_ClinicalBERT contextual token embeddings

### Few-shot

- sentence stage: fine-tune Bio_ClinicalBERT for binary sentence classification
- token stage: fine-tune Bio_ClinicalBERT for fine-grained token classification

This allows us to compare:

1. no supervised adaptation vs supervised adaptation
2. sentence stage strength vs token stage strength
3. real decomposed pipeline vs oracle sentence filtering


In [ ]:
zero_shot_prompt_rows = []
for entity_type, prompt_map in dp.SENTENCE_ZERO_SHOT_PROMPTS.items():
    zero_shot_prompt_rows.append(
        {
            "entity_type": entity_type,
            "positive_prompts": " | ".join(prompt_map["positive"]),
            "negative_prompts": " | ".join(prompt_map["negative"]),
        }
    )

display(pd.DataFrame(zero_shot_prompt_rows))


## 10. Experiment Configuration

These are the main parameters for the final run.

If a lightweight debugging pass is needed first, set `MAX_TRAIN_DOCS` and `MAX_EVAL_DOCS` to smaller values.


In [ ]:
ENTITY_TYPES = list(dp.ENTITY_TYPES)
STRATEGIES = ("zero-shot", "few-shot")
FEW_SHOT_DOCS = 40
EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 2e-5
ALLOW_DOWNLOAD = False

# Optional debug limits
MAX_TRAIN_DOCS = None
MAX_EVAL_DOCS = None

config_df = pd.DataFrame(
    [
        {"parameter": "ENTITY_TYPES", "value": ENTITY_TYPES},
        {"parameter": "STRATEGIES", "value": STRATEGIES},
        {"parameter": "FEW_SHOT_DOCS", "value": FEW_SHOT_DOCS},
        {"parameter": "EPOCHS", "value": EPOCHS},
        {"parameter": "BATCH_SIZE", "value": BATCH_SIZE},
        {"parameter": "LEARNING_RATE", "value": LEARNING_RATE},
        {"parameter": "ALLOW_DOWNLOAD", "value": ALLOW_DOWNLOAD},
        {"parameter": "MAX_TRAIN_DOCS", "value": MAX_TRAIN_DOCS},
        {"parameter": "MAX_EVAL_DOCS", "value": MAX_EVAL_DOCS},
    ]
)
display(config_df)


## 11. Run the Full Decomposed Experiments

This cell runs:

- Participants zero-shot
- Participants few-shot
- Interventions zero-shot
- Interventions few-shot
- Outcomes zero-shot
- Outcomes few-shot

and stores the results in `OUTPUT_DIR`.


In [ ]:
summary_df, artifacts = dp.run_all_decomposed_experiments(
    data_dir=DATA_DIR,
    entity_types=ENTITY_TYPES,
    strategies=STRATEGIES,
    output_dir=OUTPUT_DIR,
    few_shot_docs=FEW_SHOT_DOCS,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    seed=SEED,
    device=DEVICE,
    local_files_only=not ALLOW_DOWNLOAD,
    max_train_docs=MAX_TRAIN_DOCS,
    max_eval_docs=MAX_EVAL_DOCS,
)

display(summary_df)


## 12. Sentence-Level Results

These metrics tell us how well the first stage of the decomposed pipeline works.

If sentence F1 is high but token F1 is low, then the problem is mainly in token extraction.

If both are low, then both stages need improvement.


In [ ]:
sentence_metrics_df = summary_df[
    [
        "entity_type",
        "strategy",
        "sentence_accuracy",
        "sentence_precision",
        "sentence_recall",
        "sentence_f1",
        "sentence_keep_rate",
    ]
].sort_values(["entity_type", "strategy"]).reset_index(drop=True)

display(sentence_metrics_df)


### Sentence Confusion Matrices

To make the sentence stage easier to understand, we plot confusion matrices for each entity type and strategy.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for ax, entity_type in zip(axes, dp.ENTITY_TYPES):
    for strategy, color in [("zero-shot", "Blues"), ("few-shot", "Greens")]:
        cm = np.array(artifacts[(entity_type, strategy)].metrics["sentence_metrics"]["confusion_matrix"])
        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap=color,
            cbar=False,
            ax=ax,
        )
        ax.set_title(f"{entity_type} | {strategy}")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Gold")
        break

plt.tight_layout()
plt.show()


## 13. Token-Level Results

The token stage is reported in two ways:

1. **pipeline token metrics**: the real decomposed system
2. **oracle token metrics**: token extraction given the gold sentence filter

The difference between them is crucial.

If oracle token F1 is much higher than pipeline token F1, then sentence filtering is throwing away useful signal.

If both are low, then the token extractor itself is still weak.


In [ ]:
token_metrics_df = summary_df[
    [
        "entity_type",
        "strategy",
        "token_pipeline_precision",
        "token_pipeline_recall",
        "token_pipeline_f1",
        "token_pipeline_token_accuracy",
        "token_oracle_precision",
        "token_oracle_recall",
        "token_oracle_f1",
        "token_oracle_token_accuracy",
        "token_filter_loss",
    ]
].sort_values(["entity_type", "strategy"]).reset_index(drop=True)

display(token_metrics_df)


## 14. Main Plots

These plots summarize the whole decomposed pipeline:

1. sentence F1
2. token F1 in the real pipeline
3. token F1 with oracle sentence filtering
4. token filter loss


In [ ]:
sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 180

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.barplot(
    data=summary_df,
    x="entity_type",
    y="sentence_f1",
    hue="strategy",
    ax=axes[0, 0],
)
axes[0, 0].set_title("Sentence-Level F1")
axes[0, 0].set_ylabel("F1")
axes[0, 0].set_xlabel("")

sns.barplot(
    data=summary_df,
    x="entity_type",
    y="token_pipeline_f1",
    hue="strategy",
    ax=axes[0, 1],
)
axes[0, 1].set_title("Token-Level F1 in Decomposed Pipeline")
axes[0, 1].set_ylabel("F1")
axes[0, 1].set_xlabel("")

sns.barplot(
    data=summary_df,
    x="entity_type",
    y="token_oracle_f1",
    hue="strategy",
    ax=axes[1, 0],
)
axes[1, 0].set_title("Token-Level F1 with Oracle Filter")
axes[1, 0].set_ylabel("F1")
axes[1, 0].set_xlabel("")

sns.barplot(
    data=summary_df,
    x="entity_type",
    y="token_filter_loss",
    hue="strategy",
    ax=axes[1, 1],
)
axes[1, 1].set_title("Token Filter Loss")
axes[1, 1].set_ylabel("Oracle F1 - Pipeline F1")
axes[1, 1].set_xlabel("")

for ax in axes.flat:
    ax.legend(title="strategy")

plt.tight_layout()
plt.show()


## 15. Sentence Stage vs Token Stage Bottleneck Table

This table is designed for interpretation, not just reporting.

It shows:

- whether few-shot beats zero-shot
- whether sentence filtering is strong
- whether token extraction is strong
- how much performance is lost because of the sentence filter


In [ ]:
comparison_df = summary_df[
    [
        "entity_type",
        "strategy",
        "sentence_f1",
        "token_pipeline_f1",
        "token_oracle_f1",
        "token_filter_loss",
    ]
].copy()

comparison_df["sentence_minus_token_pipeline"] = (
    comparison_df["sentence_f1"] - comparison_df["token_pipeline_f1"]
)
comparison_df["token_problem"] = comparison_df["token_oracle_f1"] < 0.20
comparison_df["filter_problem"] = comparison_df["token_filter_loss"] > 0.05

display(comparison_df)


## 16. Preview Sentence Predictions

These examples make it easier to see what the sentence filter is doing in practice.


In [ ]:
def preview_sentence_predictions(artifact, n=5):
    rows = []
    for ex in artifact.test_sentences[:n]:
        key = (ex.doc_id, ex.sent_idx)
        rows.append(
            {
                "doc_id": ex.doc_id,
                "sent_idx": ex.sent_idx,
                "entity_type": ex.entity_type,
                "gold_keep": artifact.sentence_keep_gold[key],
                "pred_keep": artifact.sentence_keep_pred[key],
                "score": round(float(artifact.sentence_scores[key]), 4),
                "sentence": ex.text,
            }
        )
    return pd.DataFrame(rows)


display(preview_sentence_predictions(artifacts[("participants", "few-shot")], n=5))
display(preview_sentence_predictions(artifacts[("interventions", "few-shot")], n=5))
display(preview_sentence_predictions(artifacts[("outcomes", "few-shot")], n=5))


## 17. Preview Token Predictions

These examples show the actual typed BIO predictions for token extraction.


In [ ]:
def preview_token_predictions(artifact, n=5):
    rows = []
    for ex in artifact.test_sentences:
        key = (ex.doc_id, ex.sent_idx)
        pred_tags = artifact.token_sentence_pred_map[key]
        if any(tag != "O" for tag in ex.tags) or any(tag != "O" for tag in pred_tags):
            rows.append(
                {
                    "doc_id": ex.doc_id,
                    "sent_idx": ex.sent_idx,
                    "sentence": ex.text,
                    "gold_tags": " ".join(ex.tags),
                    "pred_tags": " ".join(pred_tags),
                }
            )
        if len(rows) >= n:
            break
    return pd.DataFrame(rows)


display(preview_token_predictions(artifacts[("participants", "few-shot")], n=5))
display(preview_token_predictions(artifacts[("interventions", "few-shot")], n=5))
display(preview_token_predictions(artifacts[("outcomes", "few-shot")], n=5))


## 18. Error Analysis Viewer

This cell shows mismatches in either:

- sentence keep/drop decisions
- token label sequences

This is especially useful for discussing why token identity extraction is still hard.


In [ ]:
def preview_errors(artifact, n=5):
    rows = []
    for ex in artifact.test_sentences:
        key = (ex.doc_id, ex.sent_idx)
        gold_keep = artifact.sentence_keep_gold[key]
        pred_keep = artifact.sentence_keep_pred[key]
        pred_tags = artifact.token_sentence_pred_map[key]
        if pred_keep != gold_keep or pred_tags != ex.tags:
            rows.append(
                {
                    "doc_id": ex.doc_id,
                    "sent_idx": ex.sent_idx,
                    "text": ex.text,
                    "gold_keep": gold_keep,
                    "pred_keep": pred_keep,
                    "gold_tags": " ".join(ex.tags),
                    "pred_tags": " ".join(pred_tags),
                }
            )
        if len(rows) >= n:
            break
    return pd.DataFrame(rows)


display(preview_errors(artifacts[("participants", "few-shot")], n=5))
display(preview_errors(artifacts[("interventions", "few-shot")], n=5))
display(preview_errors(artifacts[("outcomes", "few-shot")], n=5))


## 19. Build the Final PICO Tables

After sentence filtering and token extraction, we merge the predicted spans back into document-level outputs and build PICO-style tables.

Comparator recovery works as follows:

1. use intervention subtype `CONTROL` if available
2. otherwise recover comparator terms from keywords such as:
   - placebo
   - control
   - usual care
   - standard care
   - sham
   - waiting list


In [ ]:
zero_shot_pico_df = dp.build_strategy_pico_table(artifacts, "zero-shot")
few_shot_pico_df = dp.build_strategy_pico_table(artifacts, "few-shot")

print("Zero-shot PICO table")
display(zero_shot_pico_df.head(10))

print("Few-shot PICO table")
display(few_shot_pico_df.head(10))


## 20. PICO Table Coverage

This tells us how often the extracted PICO tables are actually populated.


In [ ]:
def pico_coverage(df: pd.DataFrame, strategy: str):
    columns = ["population", "intervention", "comparator", "outcome"]
    rows = []
    for col in columns:
        rows.append(
            {
                "strategy": strategy,
                "column": col,
                "non_empty_rate": (df[col].fillna("").astype(str).str.len() > 0).mean(),
            }
        )
    return pd.DataFrame(rows)


pico_coverage_df = pd.concat(
    [
        pico_coverage(zero_shot_pico_df, "zero-shot"),
        pico_coverage(few_shot_pico_df, "few-shot"),
    ],
    ignore_index=True,
)

display(pico_coverage_df)


## 21. Save the Final Result Files

The notebook saves:

- PICO tables
- metric summaries
- intermediate prediction files from the helper pipeline


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

summary_csv_path = OUTPUT_DIR / "summary_metrics.csv"
zero_shot_pico_path = OUTPUT_DIR / "zero_shot_pico_table.csv"
few_shot_pico_path = OUTPUT_DIR / "few_shot_pico_table.csv"

summary_df.to_csv(summary_csv_path, index=False)
zero_shot_pico_df.to_csv(zero_shot_pico_path, index=False)
few_shot_pico_df.to_csv(few_shot_pico_path, index=False)

print("Saved:", summary_csv_path)
print("Saved:", zero_shot_pico_path)
print("Saved:", few_shot_pico_path)


## 22. Best Configuration Summary

We identify the best overall setup according to token-level F1 in the real decomposed pipeline.


In [ ]:
best_row = summary_df.sort_values("token_pipeline_f1", ascending=False).iloc[0]
best_row_df = pd.DataFrame([best_row])
display(best_row_df)


## 23. Final Interpretation

This notebook is designed so that a reader can inspect:

1. what data exists
2. which labels are used
3. how sentence filtering is defined
4. how token tags are constructed
5. how zero-shot and few-shot differ
6. where the bottleneck lies
7. whether the final PICO table is meaningfully populated

That is why the notebook contains:

- dataset exploration
- label inspection
- alignment checks
- sentence-level examples
- token-level examples
- error analysis
- oracle vs pipeline token comparison
- saved final outputs


In [ ]:
saved_files = sorted(
    str(path.relative_to(DATA_DIR))
    for path in OUTPUT_DIR.rglob("*")
    if path.is_file()
)

display(pd.DataFrame({"saved_file": saved_files}))
